In [1]:
import pandas as pd
import vivarium_inputs
import gbd_mapping
import pathlib

Config: 'input_data:
    cache_data:
        base: True
    intermediary_data_cache_path:
        base: /share/scratch/users/{username}/cache'
Cache Dir: '/share/scratch/users/zmbc/cache'


In [2]:
location = "india"
vehicle = "rice"
scenario = "intervention"

In [3]:
# Parameters
location = "nigeria"
vehicle = "bouillon"
scenario = "intervention"

In [4]:
def aggregate_by_scenario(df):
    return (
        df.groupby(["scenario", "input_draw", "wealth_quintile"])
        .value.sum()
        .groupby(["scenario", "wealth_quintile"])
        .mean()
    )

In [5]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/person_time_anemia.parquet"
if pathlib.Path(path).is_file():
    pregnancy_person_time_anemia = pd.read_parquet(path)
else:
    pregnancy_person_time_anemia = (
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/person_time_anemia.parquet"
        )
        .assign(value=0)
        .assign(scenario=lambda x: x.scenario.replace("intervention", scenario))
    )
pregnancy_person_time_anemia

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,person_time,impairment,anemia,not_anemic,10_to_14,invalid,lowest,baseline,56,0,8.180950
1,person_time,impairment,anemia,not_anemic,10_to_14,invalid,second,baseline,56,0,11.229752
2,person_time,impairment,anemia,not_anemic,10_to_14,invalid,middle,baseline,56,0,13.414726
3,person_time,impairment,anemia,not_anemic,10_to_14,invalid,fourth,baseline,56,0,6.554923
4,person_time,impairment,anemia,not_anemic,10_to_14,invalid,highest,baseline,56,0,7.215497
...,...,...,...,...,...,...,...,...,...,...,...
719995,person_time,impairment,anemia,severe,95_plus,severe,lowest,intervention,151,0,0.000000
719996,person_time,impairment,anemia,severe,95_plus,severe,second,intervention,151,0,0.000000
719997,person_time,impairment,anemia,severe,95_plus,severe,middle,intervention,151,0,0.000000
719998,person_time,impairment,anemia,severe,95_plus,severe,fourth,intervention,151,0,0.000000


In [6]:
pregnancy_person_time_anemia.groupby("scenario").random_seed.nunique()

scenario
baseline        200
intervention    200
Name: random_seed, dtype: int64

In [7]:
pregnancy_person_time_anemia.sub_entity.value_counts()

mild          180000
moderate      180000
not_anemic    180000
severe        180000
Name: sub_entity, dtype: int64

In [8]:
total_pregnant_person_time = aggregate_by_scenario(pregnancy_person_time_anemia)
total_pregnant_person_time

scenario      wealth_quintile
baseline      fourth             1.402208e+06
              highest            1.234704e+06
              lowest             1.980982e+06
              middle             1.729295e+06
              second             2.049172e+06
intervention  fourth             1.402215e+06
              highest            1.234706e+06
              lowest             1.980992e+06
              middle             1.729301e+06
              second             2.049182e+06
Name: value, dtype: float64

In [9]:
anemic_pregnant_person_time = aggregate_by_scenario(
    pregnancy_person_time_anemia[
        pregnancy_person_time_anemia.sub_entity != "not_anemic"
    ]
)
anemic_pregnant_person_time

scenario      wealth_quintile
baseline      fourth             6.696973e+05
              highest            5.527659e+05
              lowest             1.171816e+06
              middle             9.542501e+05
              second             1.248719e+06
intervention  fourth             6.485093e+05
              highest            5.360171e+05
              lowest             1.119681e+06
              middle             9.201624e+05
              second             1.200779e+06
Name: value, dtype: float64

In [10]:
pregnant_anemia_prevalence_by_scenario = (
    anemic_pregnant_person_time / total_pregnant_person_time
).fillna(0)
pregnant_anemia_prevalence_by_scenario

scenario      wealth_quintile
baseline      fourth             0.477602
              highest            0.447691
              lowest             0.591533
              middle             0.551815
              second             0.609377
intervention  fourth             0.462489
              highest            0.434125
              lowest             0.565212
              middle             0.532101
              second             0.585980
Name: value, dtype: float64

In [11]:
path = f"./results/{location}/{vehicle}/{scenario}/pregnant_anemia_prevalence_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
pregnant_anemia_prevalence_by_scenario.to_csv(path)

In [12]:
pop = pd.read_csv(f"../0100_data_prep/results/population/stratified/{location}.csv")
pop

,sex,age_start,age_end,pregnant,wealth_quintile,value
0,Female,0.0,0.019178,not_pregnant,lowest,16960.496219
1,Female,0.0,0.019178,not_pregnant,second,16890.884668
2,Female,0.0,0.019178,not_pregnant,middle,15946.960379
3,Female,0.0,0.019178,not_pregnant,fourth,14017.956071
4,Female,0.0,0.019178,not_pregnant,highest,13029.810174
...,...,...,...,...,...,...
280,Male,95.0,125.000000,not_pregnant,lowest,1846.968043
281,Male,95.0,125.000000,not_pregnant,second,1616.329970
282,Male,95.0,125.000000,not_pregnant,middle,1661.447043
283,Male,95.0,125.000000,not_pregnant,fourth,1779.062429


In [13]:
pregnant_pop = pop[pop.pregnant == "pregnant"].groupby(["wealth_quintile"]).value.sum()
pregnant_pop

wealth_quintile
fourth     1.255512e+06
highest    1.103242e+06
lowest     1.770625e+06
middle     1.543670e+06
second     1.831699e+06
Name: value, dtype: float64

In [14]:
pregnancy_prevalent_anemia_cases_by_scenario = (
    pregnant_anemia_prevalence_by_scenario * pregnant_pop
)
pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      fourth             5.996347e+05
              highest            4.939117e+05
              lowest             1.047383e+06
              middle             8.518197e+05
              second             1.116196e+06
intervention  fourth             5.806606e+05
              highest            4.789451e+05
              lowest             1.000779e+06
              middle             8.213881e+05
              second             1.073339e+06
Name: value, dtype: float64

In [15]:
path = f"results/rescaled_pregnancy_results/{vehicle}/{location}/transition_count_maternal_disorders.parquet"
if pathlib.Path(path).is_file():
    maternal_disorders_transition_counts = pd.read_parquet(path)
else:
    maternal_disorders_transition_counts = (
        pd.read_parquet(
            f"results/rescaled_pregnancy_results/rice/india/transition_count_maternal_disorders.parquet"
        )
        .assign(value=0)
        .assign(scenario=lambda x: x.scenario.replace("intervention", scenario))
    )

maternal_disorders_transition_counts

,measure,entity_type,entity,sub_entity,age_group,anemia_status_at_birth,wealth_quintile,scenario,random_seed,input_draw,value
0,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,lowest,baseline,56,0,0.0
1,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,second,baseline,56,0,0.0
2,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,middle,baseline,56,0,0.0
3,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,fourth,baseline,56,0,0.0
4,transition_count,cause,maternal_disorders,susceptible_to_maternal_disorders_to_maternal_...,10_to_14,invalid,highest,baseline,56,0,0.0
...,...,...,...,...,...,...,...,...,...,...,...
359995,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,lowest,intervention,151,0,0.0
359996,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,second,intervention,151,0,0.0
359997,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,middle,intervention,151,0,0.0
359998,transition_count,cause,maternal_disorders,maternal_disorders_to_recovered_from_maternal_...,95_plus,severe,fourth,intervention,151,0,0.0


In [16]:
maternal_disorders_transition_counts.sub_entity.cat.categories

Index(['maternal_disorders_to_recovered_from_maternal_disorders',
       'no_transition',
       'susceptible_to_maternal_disorders_to_maternal_disorders'],
      dtype='object')

In [17]:
maternal_disorders_incident_cases_by_scenario = aggregate_by_scenario(
    maternal_disorders_transition_counts[
        maternal_disorders_transition_counts.sub_entity
        == "susceptible_to_maternal_disorders_to_maternal_disorders"
    ]
)
maternal_disorders_incident_cases_by_scenario

scenario      wealth_quintile
baseline      fourth             1.286905e+06
              highest            1.130575e+06
              lowest             1.962538e+06
              middle             1.666064e+06
              second             2.036749e+06
intervention  fourth             1.277238e+06
              highest            1.123499e+06
              lowest             1.939038e+06
              middle             1.650477e+06
              second             2.014361e+06
Name: value, dtype: float64

In [18]:
path = f"./results/{location}/{vehicle}/{scenario}/maternal_disorders_incident_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
maternal_disorders_incident_cases_by_scenario.to_csv(path)

In [19]:
path = f"results/rescaled_child_results/{vehicle}/{location}/deaths.parquet"
if pathlib.Path(path).is_file():
    neonatal_deaths = pd.read_parquet(path)
else:
    neonatal_deaths = (
        pd.read_parquet(f"results/rescaled_child_results/rice/india/deaths.parquet")
        .assign(value=0)
        .assign(
            maternal_scenario=lambda x: x.maternal_scenario.replace(
                "intervention", scenario
            )
        )
    )

neonatal_deaths = neonatal_deaths.rename(columns={"maternal_scenario": "scenario"})
neonatal_deaths

,measure,entity_type,entity,sub_entity,age_group,sex,wealth_quintile,child_scenario,scenario,random_seed,input_draw,value
0,deaths,cause,stillborn,stillborn,0_to_6_months,Female,lowest,baseline,intervention,88,0,0.000000
1,deaths,cause,stillborn,stillborn,0_to_6_months,Female,second,baseline,intervention,88,0,0.000000
2,deaths,cause,stillborn,stillborn,0_to_6_months,Female,middle,baseline,intervention,88,0,0.000000
3,deaths,cause,stillborn,stillborn,0_to_6_months,Female,fourth,baseline,intervention,88,0,0.000000
4,deaths,cause,stillborn,stillborn,0_to_6_months,Female,highest,baseline,intervention,88,0,0.000000
...,...,...,...,...,...,...,...,...,...,...,...,...
31835,deaths,cause,other_causes,other_causes,18_to_59_months,Male,lowest,baseline,baseline,64,0,145.787744
31836,deaths,cause,other_causes,other_causes,18_to_59_months,Male,second,baseline,baseline,64,0,169.643920
31837,deaths,cause,other_causes,other_causes,18_to_59_months,Male,middle,baseline,baseline,64,0,145.787744
31838,deaths,cause,other_causes,other_causes,18_to_59_months,Male,fourth,baseline,baseline,64,0,84.821960


In [20]:
neonatal_deaths_by_scenario = aggregate_by_scenario(neonatal_deaths)
neonatal_deaths_by_scenario

scenario      wealth_quintile
baseline      fourth             137459.287671
              highest            119991.265268
              lowest             204792.019852
              middle             172676.305219
              second             205783.376510
intervention  fourth             137448.684926
              highest            119924.998112
              lowest             204744.307499
              middle             172612.688749
              second             205520.958571
Name: value, dtype: float64

In [21]:
path = f"./results/{location}/{vehicle}/{scenario}/neonatal_deaths_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
neonatal_deaths_by_scenario.to_csv(path)

In [22]:
path = f"../0400_non_pregnant_anemia_model/results/{vehicle}/{location}/{scenario}/anemia_cases.parquet"
if pathlib.Path(path).is_file():
    non_pregnancy_anemia_cases = pd.read_parquet(path)
else:
    non_pregnancy_anemia_cases = (
        pd.read_parquet(
            f"../0400_non_pregnant_anemia_model/results/rice/india/intervention/anemia_cases.parquet"
        )
        .assign(value=0)
        .assign(scenario=lambda x: x.scenario.replace("intervention", scenario))
    )

non_pregnancy_anemia_cases

,sex,age_start,age_end,wealth_quintile,value,scenario
0,Female,0.0,0.019178,fourth,11611.609394,baseline
1,Female,0.0,0.019178,highest,9753.105130,baseline
2,Female,0.0,0.019178,lowest,15276.458404,baseline
3,Female,0.0,0.019178,middle,13328.773023,baseline
4,Female,0.0,0.019178,second,14808.048479,baseline
...,...,...,...,...,...,...
495,Male,95.0,125.000000,fourth,1579.569992,intervention
496,Male,95.0,125.000000,highest,1664.804547,intervention
497,Male,95.0,125.000000,lowest,1669.162311,intervention
498,Male,95.0,125.000000,middle,1472.539174,intervention


In [23]:
non_pregnancy_prevalent_anemia_cases_by_scenario = aggregate_by_scenario(
    non_pregnancy_anemia_cases.assign(entity="anemia", input_draw="draw_0")
)
non_pregnancy_prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      fourth             1.963280e+07
              highest            1.575827e+07
              lowest             2.465356e+07
              middle             2.031122e+07
              second             2.220715e+07
intervention  fourth             1.930590e+07
              highest            1.554047e+07
              lowest             2.397280e+07
              middle             1.992366e+07
              second             2.167317e+07
Name: value, dtype: float64

In [24]:
prevalent_anemia_cases_by_scenario = (
    pregnancy_prevalent_anemia_cases_by_scenario
    + non_pregnancy_prevalent_anemia_cases_by_scenario
)
prevalent_anemia_cases_by_scenario

scenario      wealth_quintile
baseline      fourth             2.023244e+07
              highest            1.625218e+07
              lowest             2.570094e+07
              middle             2.116304e+07
              second             2.332334e+07
intervention  fourth             1.988656e+07
              highest            1.601942e+07
              lowest             2.497358e+07
              middle             2.074505e+07
              second             2.274651e+07
Name: value, dtype: float64

In [25]:
path = (
    f"./results/{location}/{vehicle}/{scenario}/prevalent_anemia_cases_by_scenario.csv"
)
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
prevalent_anemia_cases_by_scenario.to_csv(path)

In [26]:
path = f"../0500_neural_tube_defects_model/results/{location}/{vehicle}/{scenario}/ntd_cases_by_scenario.csv"
if pathlib.Path(path).is_file():
    ntd_cases_by_scenario = pd.read_csv(path)
else:
    ntd_cases_by_scenario = (
        pd.read_csv(
            f"../0500_neural_tube_defects_model/results/india/rice/intervention/ntd_cases_by_scenario.csv"
        )
        .assign(value=0)
        .assign(scenario=lambda x: x.scenario.replace("intervention", scenario))
    )

ntd_cases_by_scenario = ntd_cases_by_scenario.set_index(
    ["scenario", "wealth_quintile"]
).value
ntd_cases_by_scenario

scenario      wealth_quintile
baseline      lowest             4758.801348
              second             4866.170569
              middle             4405.223395
              fourth             3904.784094
              highest            3429.644436
intervention  fourth             2280.268334
              highest            2087.552175
              lowest             1989.461541
              middle             2319.487572
              second             2148.803475
Name: value, dtype: float64

In [27]:
path = f"./results/{location}/{vehicle}/{scenario}/ntd_cases_by_scenario.csv"
pathlib.Path(path).parent.mkdir(exist_ok=True, parents=True)
ntd_cases_by_scenario.to_csv(path)